# scBIG · C2 流训练与基因空间损失

从运行的报告目录执行本 notebook。输入包括已验收的第 999 步回执、续跑日志与训练回执、正式冻结评分回执。横轴为连续的 flow microstep。每个基因损失点汇总该轮抽取的三个源域扰动上下文。

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd()
fit = json.loads((root / "training_result.json").read_text())
result = json.loads((root / "result.json").read_text())
prefix = (root / "prefix_result.json").read_bytes()
assert hashlib.sha256(prefix).hexdigest() == fit["source_prefix_sha256"]
prefix = json.loads(prefix)
raw = (root / "fit-losses.jsonl").read_bytes()
assert hashlib.sha256(raw).hexdigest() == fit["fit_history_sha256"]
rows = [json.loads(line) for line in raw.splitlines()]
steps = [row["microstep"] for row in rows]
assert steps == list(range(1000, fit["flow_microsteps"] + 1))
assert prefix["flow_microsteps"] == 999 and prefix["flow_accumulated_microsteps"] == 19
assert rows[0]["gene_space_updates"] == 3
assert rows[-1]["gene_space_updates"] == fit["gene_space_updates"]
assert fit["fit_test_conditions_accessed"] == result["test_updates"] == 0
events = [row for row in rows if row["gene_space_losses"]]
assert all(len(row["gene_space_losses"]) == 3 for row in events)

fig, axes = plt.subplots(2, 3, figsize=(12, 6.4), layout="constrained")
curve = np.array([row["ot_flow_loss"] for row in rows], dtype=float)
window = min(100, len(curve))
axes[0, 0].plot(steps, curve, lw=.8, alpha=.2, label="Each microstep", color="#32758a")
axes[0, 0].plot(steps[window-1:], np.convolve(curve, np.ones(window) / window, mode="valid"),
                lw=1.5, color="#17666e", label=f"{window}-step mean")
axes[0, 0].legend(frameon=False, fontsize=8)
axes[0, 0].set(title="OT flow objective", xlabel="Microstep", ylabel="Loss")
for axis, metric, title in (
    (axes[0, 1], "corr", "GRC correlation loss"),
    (axes[0, 2], "ot", "Reactome pathway OT loss"),
    (axes[1, 0], "total", "Weighted gene-space loss"),
):
    values = np.array([[item[metric] for item in row["gene_space_losses"]] for row in events])
    x = [row["microstep"] for row in events]
    axis.plot(x, values.mean(axis=1), marker="o", ms=3, color="#17666e")
    axis.fill_between(x, values.min(axis=1), values.max(axis=1), alpha=.17, color="#17666e")
    axis.set(title=title, xlabel="Microstep", ylabel="Mean of three source contexts")
axes[1, 1].bar(["OT flow", "Gene loss"], [fit["flow_optimizer_updates"], fit["gene_space_updates"]],
               color=["#17666e", "#b47448"])
axes[1, 1].set(title="Optimizer updates", ylabel="Count")
axes[1, 2].bar(["Resumed fit", "Frozen scoring"], [fit["fit_seconds"] / 60,
               result["evaluation_seconds"] / 60], color=["#17666e", "#566680"])
axes[1, 2].set(title="Measured runtime", ylabel="Minutes")
for axis in axes.ravel():
    axis.grid(axis="y", alpha=.18)
fig.savefig(root / "training-curves.png", dpi=175, bbox_inches="tight")
plt.show()